In [1]:
import pandas as pd
import os
from datetime import datetime, timedelta

directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2020/'

In [2]:
# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

monthly_data = []

year = 2020

for month in range(1, 13):  # Loop monthly 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:   # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
             # Load message and orderbook 
            message_df = pd.read_csv(message_file)
            orderbook_df = pd.read_csv(orderbook_file)

            # Message and orderbook data, dropper col 7
            message_df = message_df.iloc[:, :-1]  
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            base_date = filename.split('_')[1]  # Start dag SPY_2020-01-02
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merger message and orderbook data on 'Time (sec)'
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Drop NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' index for 1 second
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            # list monthly
            monthly_files.append(resampled_df)
    
    # Concatenate all daily filer for month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        # Gem
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)

# All months into one final DataFrame
final_df = pd.concat(monthly_data)

# Save the final DataFrame, year 2020
final_df.to_csv(f'combined_SPY{year}_cleaned.csv', index=False)


print(final_df.head())

<ipython-input-2-c9066cf3e83c>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  message_df = pd.read_csv(message_file)


                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2020-01-02 09:30:00    3235700.0      1700.0    3235100.0       100.0   
2020-01-02 09:30:01    3236100.0       500.0    3235700.0       500.0   
2020-01-02 09:30:02    3236100.0       600.0    3235900.0        25.0   
2020-01-02 09:30:03    3236400.0      1000.0    3236200.0       200.0   
2020-01-02 09:30:04    3236400.0      1000.0    3236100.0      1900.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2020-01-02 09:30:00    3235800.0       200.0    3235000.0      1502.0   
2020-01-02 09:30:01    3236200.0       200.0    3235500.0      1860.0   
2020-01-02 09:30:02    3236200.0      1700.0    3235700.0       235.0   
2020-01-02 09:30:03    3236500.0      1800.0    3236100.0      1500.0   
2020-01-02 09:30:04    3236500.0      1800.0    32

In [3]:
print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2020-01-02 09:30:00    3235700.0      1700.0    3235100.0       100.0   
2020-01-02 09:30:01    3236100.0       500.0    3235700.0       500.0   
2020-01-02 09:30:02    3236100.0       600.0    3235900.0        25.0   
2020-01-02 09:30:03    3236400.0      1000.0    3236200.0       200.0   
2020-01-02 09:30:04    3236400.0      1000.0    3236100.0      1900.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2020-01-02 09:30:00    3235800.0       200.0    3235000.0      1502.0   
2020-01-02 09:30:01    3236200.0       200.0    3235500.0      1860.0   
2020-01-02 09:30:02    3236200.0      1700.0    3235700.0       235.0   
2020-01-02 09:30:03    3236500.0      1800.0    3236100.0      1500.0   
2020-01-02 09:30:04    3236500.0      1800.0    32

In [4]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2020-12-31 15:59:55    3745300.0       500.0    3745100.0      1100.0   
2020-12-31 15:59:56    3744100.0       600.0    3743900.0      1100.0   
2020-12-31 15:59:57    3742800.0       400.0    3742600.0       100.0   
2020-12-31 15:59:58    3742100.0       700.0    3741900.0      2700.0   
2020-12-31 15:59:59    3741000.0      1000.0    3740800.0      2700.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2020-12-31 15:59:55    3745400.0       300.0    3744900.0       500.0   
2020-12-31 15:59:56    3744200.0       600.0    3743800.0       500.0   
2020-12-31 15:59:57    3742900.0       800.0    3742500.0      1067.0   
2020-12-31 15:59:58    3742200.0      1000.0    3741800.0      2400.0   
2020-12-31 15:59:59    3741100.0      2100.0    37

In [9]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5791398


In [10]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2020-01-02 09:30:00,3235700.0,1700.0,3235100.0,100.0,3235800.0,200.0,3235000.0,1502.0,1.0,11032372.0,100.0,3235100.0,1.0
2020-01-02 09:30:01,3236100.0,500.0,3235700.0,500.0,3236200.0,200.0,3235500.0,1860.0,1.0,11306348.0,500.0,3235700.0,1.0
2020-01-02 09:30:02,3236100.0,600.0,3235900.0,25.0,3236200.0,1700.0,3235700.0,235.0,3.0,11470044.0,10.0,3235800.0,1.0
2020-01-02 09:30:03,3236400.0,1000.0,3236200.0,200.0,3236500.0,1800.0,3236100.0,1500.0,1.0,11577220.0,1500.0,3236100.0,1.0
2020-01-02 09:30:04,3236400.0,1000.0,3236100.0,1900.0,3236500.0,1800.0,3236000.0,500.0,3.0,11621360.0,500.0,3236100.0,1.0


In [7]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2020-12-31 15:59:55,3745300.0,500.0,3745100.0,1100.0,3745400.0,300.0,3744900.0,500.0,1.0,476926768.0,1000.0,3745100.0,1.0
2020-12-31 15:59:56,3744100.0,600.0,3743900.0,1100.0,3744200.0,600.0,3743800.0,500.0,3.0,477087504.0,300.0,3744100.0,-1.0
2020-12-31 15:59:57,3742800.0,400.0,3742600.0,100.0,3742900.0,800.0,3742500.0,1067.0,1.0,477255304.0,200.0,3742500.0,1.0
2020-12-31 15:59:58,3742100.0,700.0,3741900.0,2700.0,3742200.0,1000.0,3741800.0,2400.0,3.0,477406088.0,200.0,3741900.0,1.0
2020-12-31 15:59:59,3741000.0,1000.0,3740800.0,2700.0,3741100.0,2100.0,3740700.0,2600.0,3.0,477571748.0,200.0,3740700.0,1.0


In [8]:
# Save 
final_df.to_csv('final_combined_2020.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2020_with_time.csv', index=True)

In [11]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 253


In [12]:
# Load the data from 'final_combined_2020_with_time.csv'
final_combined_2020_with_time = pd.read_csv('final_combined_2020_with_time.csv')


final_combined_2020_with_time['Time (sec)'] = pd.to_datetime(final_combined_2020_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2020_with_time.groupby(final_combined_2020_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate 
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

In [13]:
print(resampled_5min_final_df.head(10))

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2020-01-02 09:30:00          3235700.0        3238600.0        3234200.0   
1 2020-01-02 09:35:00          3236800.0        3239800.0        3236600.0   
2 2020-01-02 09:40:00          3238500.0        3240000.0        3238000.0   
3 2020-01-02 09:45:00          3238700.0        3240300.0        3237400.0   
4 2020-01-02 09:50:00          3237500.0        3238600.0        3235800.0   
5 2020-01-02 09:55:00          3236300.0        3239200.0        3236000.0   
6 2020-01-02 10:00:00          3238700.0        3239000.0        3234900.0   
7 2020-01-02 10:05:00          3235600.0        3236300.0        3235000.0   
8 2020-01-02 10:10:00          3235800.0        3236300.0        3231200.0   
9 2020-01-02 10:15:00          3231300.0        3231300.0        3227300.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         3237100.0          3235100.0        3238500.0        32

In [14]:
# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2020_corrected.csv', index=False)

In [15]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2020-01-02    78
2020-01-03    78
2020-01-06    78
2020-01-07    78
2020-01-08    78
              ..
2020-12-24    42
2020-12-28    78
2020-12-29    78
2020-12-30    78
2020-12-31    78
Length: 253, dtype: int64


In [16]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2020_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2020-01-02 09:30:00          3235700.0        3238600.0        3234200.0   
1 2020-01-02 09:35:00          3236800.0        3239800.0        3236600.0   
2 2020-01-02 09:40:00          3238500.0        3240000.0        3238000.0   
3 2020-01-02 09:45:00          3238700.0        3240300.0        3237400.0   
4 2020-01-02 09:50:00          3237500.0        3238600.0        3235800.0   
5 2020-01-02 09:55:00          3236300.0        3239200.0        3236000.0   
6 2020-01-02 10:00:00          3238700.0        3239000.0        3234900.0   
7 2020-01-02 10:05:00          3235600.0        3236300.0        3235000.0   
8 2020-01-02 10:10:00          3235800.0        3236300.0        3231200.0   
9 2020-01-02 10:15:00          3231300.0        3231300.0        3227300.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         3237100.0          3235100.0        3238500.0        32

In [17]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2020_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2020-01-02 09:30:00,3235700.0,3238600.0,3234200.0,3237100.0,3235100.0,3238500.0,3234100.0,3237000.0,3235800.0,3238700.0,3234300.0,3237200.0,3235000.0,3238400.0,3234000.0,3236900.0,283813.0,946.043333,336773.0,1122.576667,385465.0,1284.883333,387584.0,1291.946667,3235100.0,3238600.0,3234100.0,3237000.0,-0.013333,2020-01-02
2020-01-02 09:35:00,3236800.0,3239800.0,3236600.0,3238500.0,3236700.0,3239600.0,3236500.0,3238400.0,3236900.0,3239900.0,3236700.0,3238600.0,3236600.0,3239500.0,3236400.0,3238300.0,472161.0,1573.870000,243970.0,813.233333,502472.0,1674.906667,385094.0,1283.646667,3236800.0,3239800.0,3236600.0,3238400.0,0.060000,2020-01-02
2020-01-02 09:40:00,3238500.0,3240000.0,3238000.0,3238900.0,3238400.0,3239800.0,3237900.0,3238800.0,3238600.0,3240100.0,3238100.0,3239000.0,3238300.0,3239700.0,3237800.0,3238700.0,324899.0,1086.618729,225462.0,754.053512,303234.0,1014.160535,294823.0,986.030100,3238500.0,3239900.0,3237900.0,3238800.0,-0.043478,2020-01-02
2020-01-02 09:45:00,3238700.0,3240300.0,3237400.0,3237600.0,3238500.0,3240100.0,3237300.0,3237400.0,3238800.0,3240400.0,3237500.0,3237700.0,3238400.0,3240000.0,3237200.0,3237300.0,320824.0,1069.413333,235789.0,785.963333,352086.0,1173.620000,304884.0,1016.280000,3238700.0,3240300.0,3237300.0,3237400.0,0.066667,2020-01-02
2020-01-02 09:50:00,3237500.0,3238600.0,3235800.0,3236300.0,3237400.0,3238500.0,3235700.0,3236100.0,3237600.0,3238700.0,3235900.0,3236400.0,3237300.0,3238400.0,3235600.0,3236000.0,298513.0,995.043333,248803.0,829.343333,334414.0,1114.713333,294782.0,982.606667,3237500.0,3238700.0,3235600.0,3236300.0,0.173333,2020-01-02
2020-01-02 09:55:00,3236300.0,3239200.0,3236000.0,3238800.0,3236200.0,3239000.0,3235800.0,3238700.0,3236400.0,3239300.0,3236100.0,3238900.0,3236100.0,3238900.0,3235700.0,3238600.0,230323.0,767.743333,257372.0,857.906667,252607.0,842.023333,308173.0,1027.243333,3236200.0,3239200.0,3235800.0,3238700.0,0.180000,2020-01-02
2020-01-02 10:00:00,3238700.0,3239000.0,3234900.0,3235800.0,3238600.0,3238900.0,3234800.0,3235700.0,3238800.0,3239100.0,3235000.0,3235900.0,3238500.0,3238800.0,3234700.0,3235600.0,345383.0,1151.276667,232123.0,773.743333,379908.0,1266.360000,235527.0,785.090000,3238600.0,3239000.0,3234800.0,3235750.0,-0.026667,2020-01-02
2020-01-02 10:05:00,3235600.0,3236300.0,3235000.0,3235800.0,3235500.0,3236200.0,3234900.0,3235700.0,3235700.0,3236400.0,3235100.0,3235900.0,3235400.0,3236100.0,3234800.0,3235600.0,201776.0,672.586667,247503.0,825.010000,244988.0,816.626667,272423.0,908.076667,3235500.0,3236300.0,3234800.0,3235800.0,0.066667,2020-01-02
2020-01-02 10:10:00,3235800.0,3236300.0,3231200.0,3231300.0,3235700.0,3236200.0,3231000.0,3231200.0,3235900.0,3236400.0,3231300.0,3231400.0,3235600.0,3236100.0,3230900.0,3231100.0,312187.0,1040.623333,255474.0,851.580000,308046.0,1026.820000,284864.0,949.546667,3235700.0,3236300.0,3230900.0,3231250.0,0.113333,2020-01-02
2020-01-02 10:15:00,3231300.0,3231300.0,3227300.0,3229500.0,3231100.0,3231100.0,3227200.0,3229400.0,3231400.0,3231400.0,3227400.0,3229600.0,3231000.0,3231000.0,3227100.0,3229300.0,280208.0,934.026667,242995.0,809.983333,260031.0,866.770000,275676.0,918.920000,3231100.0,3231100.0,3227200.0,3229400.0,0.126667,2020-01-02


In [18]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.tail(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2020_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2020-12-31 15:10:00,3732200.0,3735600.0,3731700.0,3734400.0,3732100.0,3735500.0,3731500.0,3734300.0,3732300.0,3735700.0,3731800.0,3734500.0,3732000.0,3735400.0,3731400.0,3734200.0,163906.0,546.353333,191580.0,638.600000,498068.0,1660.226667,349619.0,1165.396667,3732100.0,3735600.0,3731500.0,3734500.0,0.120000,2020-12-31
2020-12-31 15:15:00,3734400.0,3736400.0,3734400.0,3735400.0,3734300.0,3736300.0,3734300.0,3735200.0,3734500.0,3736500.0,3734500.0,3735500.0,3734200.0,3736200.0,3734200.0,3735100.0,305368.0,1017.893333,206221.0,687.403333,578317.0,1927.723333,292257.0,974.190000,3734300.0,3736400.0,3734300.0,3735300.0,0.060000,2020-12-31
2020-12-31 15:20:00,3735300.0,3735400.0,3733900.0,3734200.0,3735200.0,3735200.0,3733800.0,3734100.0,3735400.0,3735500.0,3734000.0,3734300.0,3735100.0,3735100.0,3733700.0,3734000.0,370931.0,1244.734899,204004.0,684.577181,671489.0,2253.318792,287993.0,966.419463,3735400.0,3735400.0,3733800.0,3734200.0,0.127517,2020-12-31
2020-12-31 15:25:00,3734100.0,3735100.0,3733100.0,3733400.0,3734000.0,3735000.0,3733000.0,3733300.0,3734200.0,3735200.0,3733200.0,3733500.0,3733900.0,3734900.0,3732900.0,3733200.0,199419.0,664.730000,261474.0,871.580000,360517.0,1201.723333,417882.0,1392.940000,3734200.0,3735200.0,3733000.0,3733400.0,0.020000,2020-12-31
2020-12-31 15:30:00,3733300.0,3735100.0,3732100.0,3734400.0,3733200.0,3735000.0,3732000.0,3734300.0,3733400.0,3735200.0,3732200.0,3734500.0,3733100.0,3734900.0,3731900.0,3734200.0,228857.0,762.856667,267428.0,891.426667,401584.0,1338.613333,368404.0,1228.013333,3733300.0,3735100.0,3732000.0,3734300.0,0.000000,2020-12-31
2020-12-31 15:35:00,3734500.0,3736700.0,3733700.0,3733900.0,3734400.0,3736600.0,3733500.0,3733700.0,3734600.0,3736800.0,3733800.0,3734000.0,3734300.0,3736500.0,3733400.0,3733600.0,268461.0,894.870000,149160.0,497.200000,420683.0,1402.276667,329287.0,1097.623333,3734400.0,3736700.0,3733400.0,3733900.0,0.160000,2020-12-31
2020-12-31 15:40:00,3733900.0,3734000.0,3730700.0,3732900.0,3733800.0,3733900.0,3730500.0,3732800.0,3734000.0,3734100.0,3730800.0,3733000.0,3733700.0,3733800.0,3730400.0,3732700.0,258048.0,860.160000,185305.0,617.683333,425679.0,1418.930000,361624.0,1205.413333,3733800.0,3734100.0,3730600.0,3732800.0,0.100000,2020-12-31
2020-12-31 15:45:00,3732900.0,3736600.0,3732500.0,3736300.0,3732700.0,3736400.0,3732400.0,3736200.0,3733000.0,3736700.0,3732600.0,3736400.0,3732600.0,3736300.0,3732300.0,3736100.0,356284.0,1191.585284,194905.0,651.856187,602578.0,2015.311037,398183.0,1331.715719,3732800.0,3736600.0,3732400.0,3736300.0,0.050167,2020-12-31
2020-12-31 15:50:00,3734500.0,3742500.0,3734500.0,3740600.0,3734300.0,3742400.0,3734300.0,3740500.0,3734600.0,3742600.0,3734600.0,3740700.0,3734200.0,3742300.0,3734200.0,3740400.0,194263.0,647.543333,302214.0,1007.380000,343727.0,1145.756667,447970.0,1493.233333,3734300.0,3742500.0,3734300.0,3740400.0,0.140000,2020-12-31
2020-12-31 15:55:00,3740700.0,3746400.0,3738600.0,3741000.0,3740500.0,3746200.0,3738500.0,3740800.0,3740800.0,3746500.0,3738700.0,3741100.0,3740400.0,3746100.0,3738400.0,3740700.0,259296.0,864.320000,422986.0,1409.953333,487583.0,1625.276667,527302.0,1757.673333,3740700.0,3746400.0,3738600.0,3740700.0,-0.066667,2020-12-31
